# Making the full RAG system using what we have done till now and connecting with llm so that it generates the answer

In [1]:
# We need three components for our basic RAG system:
# 1. Embedding model → converts the question into a vector
# 2. ChromaDB → retrieves relevant document chunks
# 3. Groq LLM → generates the final answer

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

# Load the same embedding model used when creating ChromaDB.
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Connect to our existing ChromaDB.
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embedding_model
)

# Initialize our Groq LLM.
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0
)

print("Embedding model loaded.")
print("ChromaDB connected.")
print("LLM loaded.")

d:\VeriRAG_Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2230.58it/s]


Embedding model loaded.
ChromaDB connected.
LLM loaded.


In [2]:
# This is the RETRIEVAL part of RAG.

# We retrieve the top 3 chunks related to our question.
# Then we combine those chunks into one context string
# that will later be given to the LLM.

query = "How does the Transformer architecture use attention?"

results = vectorstore.similarity_search(
    query,
    k=3
)

# Combine the retrieved chunks into one context.
context = "\n\n".join(
    result.page_content
    for result in results
)

print("Retrieved chunks:", len(results))

print("CONTEXT")

print(context[:3000])

Retrieved chunks: 3
CONTEXT
The Transformer uses multi-head attention in three different ways:
• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,
and the memory keys and values come from the output of the encoder. This allows every
position in the decoder to attend over all positions in the input sequence. This mimics the
typical encoder-decoder attention mechanisms in sequence-to-sequence models such as
[38, 2, 9].
• The encoder contains self-attention layers. In a self-attention layer all of the keys, values
and queries come from the same place, in this case, the output of the previous layer in the
encoder. Each position in the encoder can attend to all positions in the previous layer of the
encoder.
• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to
all positions in the decoder up to and including that position. We need to prevent leftward

collected through web sources. This data contains private

In [3]:
# Now we perform the GENERATION part of RAG.

# Instead of asking the LLM to answer only from its
# pretrained knowledge, we explicitly provide the retrieved document context.

prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the information
provided in the context below.

If the answer cannot be found in the context,
say that the information is not available in the
provided documents.

Context:
--------------------
{context}
--------------------

Question:
{query}

Answer:
"""

# Send the prompt containing both the context and question
# to the Groq language model.
response = llm.invoke(prompt)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(response.content)

QUESTION:
How does the Transformer architecture use attention?

ANSWER:
Based on the provided context, the Transformer architecture uses multi-head attention in three different ways:

1.  **Encoder-Decoder Attention:** In these layers, queries come from the previous decoder layer, while memory keys and values come from the output of the encoder. This allows every position in the decoder to attend to all positions in the input sequence.
2.  **Encoder Self-Attention:** The encoder contains self-attention layers where keys, values, and queries all come from the same source (the output of the previous layer in the encoder). This allows each position in the encoder to attend to all positions in the previous encoder layer.
3.  **Decoder Self-Attention:** The decoder uses self-attention layers that allow each position in the decoder to attend to all positions in the decoder up to and including that position.

Additionally, the context notes that the encoder is composed of a stack of identical